# Download World Bank Data Tables

This notebook downloads all World Bank indicator data and saves each as a separate table in the `worldbank` schema.
Each table's description is set using the long description from World Bank metadata.

Uses joblib Parallel for multi-threaded parallel downloads.

In [ ]:
%pip install wbgapi tqdm requests joblib pandas --quiet

In [ ]:
# Configuration
CATALOG = "main_catalog"
SCHEMA = "worldbank"

# Number of parallel jobs (threads)
N_JOBS = 8

# Set to True to drop all existing tables and start fresh
FRESH_START = False

# Limit number of indicators to process (None for all, or set a number for testing)
INDICATOR_LIMIT = None

In [ ]:
import wbgapi as wb
import requests
import pandas as pd
import re
import contextlib
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
import threading
import joblib

@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager to patch joblib to report into tqdm progress bar."""
    
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()

In [ ]:
def sanitize_table_name(indicator_id: str) -> str:
    """Convert indicator ID to a valid table name.
    
    World Bank indicator IDs like 'SP.POP.TOTL' become 'sp_pop_totl'
    """
    name = indicator_id.lower()
    name = re.sub(r'[^a-z0-9_]', '_', name)
    name = re.sub(r'_+', '_', name)
    name = name.strip('_')
    
    if name[0].isdigit():
        name = 'ind_' + name
    
    return name[:128]

In [ ]:
def fetch_indicator_metadata(indicator_id: str) -> dict:
    """Fetch full metadata from World Bank API directly."""
    url = f"https://api.worldbank.org/v2/indicator/{indicator_id}?format=json"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    
    if len(data) < 2 or not data[1]:
        return None
    
    return data[1][0]

In [ ]:
def fetch_indicator_data(indicator_id: str) -> pd.DataFrame:
    """Fetch all data for an indicator using wbgapi.
    
    Returns a DataFrame with columns: country_code, country_name, year, value
    """
    try:
        # Fetch data - omit economy parameter to get all countries
        df = wb.data.DataFrame(
            indicator_id,
            time=range(1960, 2025),
            labels=True,
            skipBlanks=True,
            columns='time'
        )
        
        if df.empty:
            return pd.DataFrame()
        
        df = df.reset_index()
        
        # Identify year columns (YR2020, YR2021, etc.)
        year_cols = [col for col in df.columns if str(col).startswith('YR')]
        id_vars = [col for col in df.columns if not str(col).startswith('YR')]
        
        if not year_cols:
            return pd.DataFrame()
        
        # Melt from wide to long format
        df_melted = df.melt(
            id_vars=id_vars,
            value_vars=year_cols,
            var_name='year',
            value_name='value'
        )
        
        # Convert year column from 'YR2020' to 2020
        df_melted['year'] = df_melted['year'].str.replace('YR', '').astype(int)
        df_melted = df_melted.dropna(subset=['value'])
        
        # Standardize column names
        if 'economy' in df_melted.columns:
            df_melted = df_melted.rename(columns={'economy': 'country_code'})
        if 'Country' in df_melted.columns:
            df_melted = df_melted.rename(columns={'Country': 'country_name'})
        
        # Ensure required columns exist
        if 'country_code' not in df_melted.columns:
            df_melted['country_code'] = ''
        if 'country_name' not in df_melted.columns:
            df_melted['country_name'] = df_melted.get('country_code', '')
        
        return df_melted[['country_code', 'country_name', 'year', 'value']]
        
    except Exception as e:
        return pd.DataFrame()

In [ ]:
# Create schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"""
    COMMENT ON SCHEMA {CATALOG}.{SCHEMA} IS 
    'World Bank indicator data tables. Each table contains time series data for a specific indicator across countries.'
""")
print(f"Schema {CATALOG}.{SCHEMA} ready")

if FRESH_START:
    # Drop all tables in the schema
    tables_df = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}")
    tables = [row.tableName for row in tables_df.collect()]
    for table in tables:
        spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA}.{table}")
    print(f"Dropped {len(tables)} tables for fresh start")

In [ ]:
# Get existing tables to support resume
existing_tables = set()
try:
    tables_df = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}")
    existing_tables = set(row.tableName for row in tables_df.collect())
    print(f"Found {len(existing_tables)} existing tables")
except Exception as e:
    print(f"No existing tables found: {e}")

In [ ]:
# Get list of all indicators
print("Fetching indicator list...")
series_list = wb.series.info()
all_indicators = [(s.get("id"), s.get("value", "")) for s in series_list.items]
print(f"Total indicators: {len(all_indicators)}")

# Filter out already loaded indicators
indicators_to_load = [
    (ind_id, ind_name) for ind_id, ind_name in all_indicators 
    if sanitize_table_name(ind_id) not in existing_tables
]

print(f"Already loaded: {len(all_indicators) - len(indicators_to_load)}")
print(f"Remaining to load: {len(indicators_to_load)}")

if INDICATOR_LIMIT:
    indicators_to_load = indicators_to_load[:INDICATOR_LIMIT]
    print(f"Limited to first {INDICATOR_LIMIT} indicators")

In [ ]:
# Define schema for data tables
data_schema = StructType([
    StructField("country_code", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("value", DoubleType(), True),
])

# Thread-safe counter for progress tracking
progress_lock = threading.Lock()
progress = {'success': 0, 'skipped': 0, 'error': 0}

In [ ]:
def process_indicator(indicator_id: str, indicator_name: str) -> dict:
    """Download data for one indicator and save as a table.
    
    Returns a dict with status info.
    """
    table_name = sanitize_table_name(indicator_id)
    full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    
    result = {
        'indicator_id': indicator_id,
        'table_name': table_name,
        'status': 'unknown',
        'rows': 0,
        'error': None
    }
    
    try:
        # Fetch the data
        df_pandas = fetch_indicator_data(indicator_id)
        
        if df_pandas.empty:
            result['status'] = 'skipped'
            result['error'] = 'No data available'
            with progress_lock:
                progress['skipped'] += 1
            return result
        
        # Fetch metadata for the long description
        metadata = fetch_indicator_metadata(indicator_id)
        long_description = ""
        if metadata:
            long_description = metadata.get('sourceNote', '') or ''
        
        # Truncate description if too long for SQL comment
        if len(long_description) > 4000:
            long_description = long_description[:3997] + "..."
        
        # Escape single quotes in description for SQL
        long_description_escaped = long_description.replace("'", "''")
        
        # Convert to Spark DataFrame
        df_spark = spark.createDataFrame(df_pandas, schema=data_schema)
        
        # Write to Delta table
        df_spark.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(full_table_name)
        
        # Set table comment with the long description
        if long_description_escaped:
            spark.sql(f"COMMENT ON TABLE {full_table_name} IS '{long_description_escaped}'")
        
        result['status'] = 'success'
        result['rows'] = len(df_pandas)
        
        with progress_lock:
            progress['success'] += 1
        
    except Exception as e:
        result['status'] = 'error'
        result['error'] = str(e)
        with progress_lock:
            progress['error'] += 1
    
    return result

In [ ]:
# Process all indicators in parallel using joblib with tqdm progress bar
print(f"Processing {len(indicators_to_load)} indicators with {N_JOBS} parallel jobs...")

with tqdm_joblib(tqdm(desc="Downloading indicators", total=len(indicators_to_load))):
    results = Parallel(n_jobs=N_JOBS, backend='threading')(
        delayed(process_indicator)(ind_id, ind_name) 
        for ind_id, ind_name in indicators_to_load
    )

print()
print("="*50)
print("Processing complete!")
print(f"  Success: {progress['success']}")
print(f"  Skipped (no data): {progress['skipped']}")
print(f"  Errors: {progress['error']}")

In [ ]:
# Show any errors
errors = [r for r in results if r['status'] == 'error']
if errors:
    print(f"\nErrors encountered ({len(errors)}):")
    for e in errors[:10]:  # Show first 10 errors
        print(f"  {e['indicator_id']}: {e['error']}")
    if len(errors) > 10:
        print(f"  ... and {len(errors) - 10} more")

In [ ]:
# Summary statistics
successful = [r for r in results if r['status'] == 'success']
total_rows = sum(r['rows'] for r in successful)

print(f"\n=== Summary ===")
print(f"Tables created: {len(successful)}")
print(f"Total rows across all tables: {total_rows:,}")
print(f"Average rows per table: {total_rows // max(len(successful), 1):,}")

In [ ]:
# Verify by listing some tables in the schema
tables_df = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}")
print(f"\nTotal tables in {CATALOG}.{SCHEMA}: {tables_df.count()}")
print("\nSample tables:")
display(tables_df.limit(10))

In [ ]:
# Show a sample table with its description
if successful:
    sample = successful[0]
    sample_table = f"{CATALOG}.{SCHEMA}.{sample['table_name']}"
    print(f"\nSample table: {sample_table}")
    print(f"Rows: {sample['rows']}")
    
    # Show table description
    desc = spark.sql(f"DESCRIBE TABLE EXTENDED {sample_table}")
    comment_row = [row for row in desc.collect() if row.col_name == 'Comment']
    if comment_row:
        print(f"\nDescription: {comment_row[0].data_type[:500]}...")
    
    print("\nSample data:")
    display(spark.sql(f"SELECT * FROM {sample_table} LIMIT 5"))